# 05 · Field-risk ablation on the supplied dataset

This notebook compares the field detector's evidence groups using the current `field_classification_*.csv` labels and matching payload values. It evaluates only the held-out `test` split. GLiNER text-span metrics are kept in notebook 04 because they are a separate task and use token-span labels rather than field-level labels.

Configurations: validators only; validators with a drift-adjusted threshold; field classifier only; classifier with a drift-adjusted threshold; and a combined validator/classifier decision. Drift is not treated as proof of sensitivity; it only changes the review threshold.

In [ ]:
import csv, json, hashlib, platform, sys, time, collections
from pathlib import Path
import numpy as np

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "data" / "synthetic" / "field_classification_train.csv").exists()), HERE)
DATA = ROOT / "data" / "synthetic"
MODELS, RESULTS = ROOT / "models", ROOT / "results"
MODELS.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
try:
    from app.detectors import validators as V
    from app.detectors.field_classifier import ClassifierConfig, FieldObservation, FieldRiskClassifier
except ImportError:
    # Local development fallback when the detector files are beside the notebook.
    import validators as V
    from field_classifier import ClassifierConfig, FieldObservation, FieldRiskClassifier

# The current CSV contains engineered audit columns, while FieldRiskClassifier
# expects raw FieldObservation inputs. Rebuild each observation from the matching
# payload value and retain only labels/split/drift metadata from the CSV.
def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]
def flatten_values(obj, prefix=""):
    out = {}
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            out.update(flatten_values(value, path))
    elif isinstance(obj, list):
        for value in obj: out.update(flatten_values(value, prefix + "[]"))
    else:
        out[prefix] = obj
    return out
payload_rows = read_jsonl(DATA / "baseline_payloads.jsonl") + read_jsonl(DATA / "drift_payloads.jsonl")
payload_by_id = {r["record_id"]: r for r in payload_rows}

def drift_category(row):
    if int(row.get("is_new_field", 0)): return "NEW_FIELD"
    if int(row.get("is_renamed_field", 0)): return "RENAME_MATCH"
    if int(row.get("has_type_shift", 0)): return "TYPE_SHIFT"
    if int(row.get("has_distribution_shift", 0)): return "DISTRIBUTION_SHIFT"
    return "NO_DRIFT"

def load_field_split(split):
    path = DATA / f"field_classification_{split}.csv"
    with path.open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    result = []
    missing = []
    for row in rows:
        rec = payload_by_id.get(row["record_id"])
        if rec is None:
            missing.append(row["record_id"]); continue
        values = flatten_values(rec["payload"])
        field_path = row["field_path"]
        if field_path not in values:
            missing.append(f"{row['record_id']}:{field_path}"); continue
        value = values[field_path]
        obs = FieldObservation(field_path=field_path, values=[value], drift_category=drift_category(row))
        is_text = (row.get("field_name", "").lower() in {"notes", "message", "description", "comment", "clinical_note", "support_text"}
                   or (isinstance(value, str) and len(value) >= 48))
        result.append({"record_id": row["record_id"], "group_id": row.get("group_id", row["record_id"]),
                       "field_path": field_path, "field_name": row.get("field_name", field_path.split(".")[-1]),
                       "value": value, "observation": obs, "label": int(row["label_sensitive"]),
                       "drift_label": int(row["label_drift_anomaly"]), "drift_category": obs.drift_category,
                       "is_text": bool(is_text), "category": row.get("category", "") or "",
                       "is_new_field": int(row.get("is_new_field", 0)),
                       "is_renamed_field": int(row.get("is_renamed_field", 0)),
                       "has_type_shift": int(row.get("has_type_shift", 0)),
                       "has_distribution_shift": int(row.get("has_distribution_shift", 0))})
    if missing:
        raise ValueError(f"{split}: {len(missing)} field rows could not be matched to payload values; examples: {missing[:5]}")
    return result

splits = {s: load_field_split(s) for s in ("train", "validation", "test")}
print("Project root:", ROOT, "| machine:", platform.machine())
print("Field rows:", {s: len(v) for s, v in splits.items()})

## Fit the full classifier on training split and extract test evidence

In [ ]:
SEED = 13
clf = FieldRiskClassifier(ClassifierConfig(seed=SEED)).fit(
    [r["observation"] for r in splits["train"]],
    [r["label"] for r in splits["train"]]
)

def metrics(y, pred):
    y, pred = np.asarray(y, bool), np.asarray(pred, bool)
    tp=int((y&pred).sum()); fp=int((~y&pred).sum()); fn=int((y&~pred).sum()); tn=int((~y&~pred).sum())
    p=tp/(tp+fp) if tp+fp else 0.0; r=tp/(tp+fn) if tp+fn else 0.0
    return {"n":len(y),"precision":p,"recall":r,"f1":2*p*r/(p+r) if p+r else 0.0,
            "fnr":fn/(tp+fn) if tp+fn else 0.0,"fpr":fp/(fp+tn) if fp+tn else 0.0,
            "tp":tp,"fp":fp,"fn":fn,"tn":tn}

test_rows = splits["test"]
y = np.asarray([r["label"] for r in test_rows], dtype=int)
t0 = time.perf_counter(); probabilities = clf.predict_proba([r["observation"] for r in test_rows]); clf_ms=(time.perf_counter()-t0)*1000
validator_scores=[]; validator_ms=[]
for r in test_rows:
    t0=time.perf_counter()
    ratios=V.field_hit_ratios([r["value"]], embedded=r["is_text"])
    validator_scores.append(max(ratios.values(), default=0.0)); validator_ms.append((time.perf_counter()-t0)*1000)
validator_scores=np.asarray(validator_scores)
print("Test rows:",len(test_rows),"| sensitive positives:",int(y.sum()),"| prevalence:",float(y.mean()))
print("Classifier inference ms total:",round(clf_ms,2),"| validator median ms:",round(float(np.median(validator_ms)),5))

## Compare configurations

In [ ]:
def has_drift(r): return r["drift_category"] != "NO_DRIFT"

def flag_a(i): return validator_scores[i] >= 0.40
def flag_b(i): return validator_scores[i] >= (0.10 if has_drift(test_rows[i]) else 0.40)
def flag_c(i): return probabilities[i] >= 0.50
def flag_d(i): return probabilities[i] >= (0.40 if has_drift(test_rows[i]) else 0.70)
def flag_e(i): return flag_b(i) or flag_d(i)
CONFIGS={"A validators":flag_a,"B validators + drift threshold":flag_b,
         "C classifier p>=0.50":flag_c,"D classifier + drift threshold":flag_d,
         "E combined":flag_e}
results={"environment":{"python":platform.python_version(),"machine":platform.machine()},
         "classifier_version":clf.model_version,"n_test_fields":len(test_rows),"test_positive_rate":float(y.mean()),
         "field_level":{},"payload_level":{}}
print(f"{'configuration':<31}{'P':>7}{'R':>7}{'F1':>7}{'FNR':>7}{'FPR':>7}{'n':>7}")
for name, fn in CONFIGS.items():
    pred=np.asarray([fn(i) for i in range(len(test_rows))],dtype=bool)
    m=metrics(y,pred); results["field_level"][name]=m
    print(f"{name:<31}{m['precision']:>7.3f}{m['recall']:>7.3f}{m['f1']:>7.3f}{m['fnr']:>7.3f}{m['fpr']:>7.3f}{m['n']:>7}")
print("A and B identical on this split:", all(flag_a(i)==flag_b(i) for i in range(len(test_rows))))
print("C and D identical on this split:", all(flag_c(i)==flag_d(i) for i in range(len(test_rows))))

## Results by field novelty/drift group

These groups are defined from the supplied dataset's metadata. They are evaluation slices, not claims that the model was tested on a separately constructed unseen-name benchmark.

In [ ]:
groups={
    "all": list(range(len(test_rows))),
    "new_field": [i for i,r in enumerate(test_rows) if r["is_new_field"]],
    "renamed_field": [i for i,r in enumerate(test_rows) if r["is_renamed_field"]],
    "type_shift": [i for i,r in enumerate(test_rows) if r["has_type_shift"]],
    "distribution_shift": [i for i,r in enumerate(test_rows) if r["has_distribution_shift"]],
    "no_drift": [i for i,r in enumerate(test_rows) if r["drift_category"]=="NO_DRIFT"],
}
for group, idx in groups.items():
    results.setdefault("by_group", {})[group]={}
    if not idx: continue
    for name, fn in CONFIGS.items():
        pred=[fn(i) for i in idx]
        results["by_group"][group][name]=metrics(y[idx],pred)
    print(group,"n=",len(idx),"sensitive=",int(y[idx].sum()),"combined=",results["by_group"][group]["E combined"])

## Payload-level leakage proxy

This proxy asks whether at least one sensitive field in a payload would be left unflagged by a configuration. It is **not** a measurement of actual downstream leakage; the real enforcement path must redact/quarantine and revalidate.

In [ ]:
by_record=collections.defaultdict(list)
for i,r in enumerate(test_rows): by_record[r["record_id"]].append(i)
for name, fn in CONFIGS.items():
    sensitive_payloads=[]; leaks=[]; overflags=[]
    for rec_id, idx in by_record.items():
        truth=[bool(y[i]) for i in idx]; pred=[bool(fn(i)) for i in idx]
        if any(truth):
            sensitive_payloads.append(rec_id)
            leaks.append(any(t and not p for t,p in zip(truth,pred)))
        if any(not t for t in truth):
            overflags.append(any((not t) and p for t,p in zip(truth,pred)))
    summary={"payload_count":len(by_record),"sensitive_payloads":len(sensitive_payloads),
             "payload_leak_proxy_rate":float(np.mean(leaks)) if leaks else 0.0,
             "payload_overflag_proxy_rate":float(np.mean(overflags)) if overflags else 0.0}
    results["payload_level"][name]=summary
    print(name,summary)

## Save results and explainability summary

In [ ]:
results["latency_ms"]={"classifier_total_test_ms":float(clf_ms),
                       "validator_p50_ms_per_field":float(np.percentile(validator_ms,50)) if validator_ms else None,
                       "validator_p95_ms_per_field":float(np.percentile(validator_ms,95)) if validator_ms else None}
results["numeric_weights_top"] = dict(sorted(clf.numeric_weights().items(), key=lambda kv:-abs(kv[1]))[:15])
results["caveats"]=["Synthetic labels and scenarios", "Field-level flags are not policy enforcement", "No dedicated unseen-name split in supplied dataset", "Latency is from this notebook environment; rerun on LinuxONE for s390x evidence"]
out=RESULTS/"ablation.json"
out.write_text(json.dumps(results,indent=2),encoding="utf-8")
print("Saved",out)
print("Largest numeric weights:")
for k,v in results["numeric_weights_top"].items(): print(f"  {k:<30}{v:+.3f}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    names=list(CONFIGS)
    fig,ax=plt.subplots(figsize=(9,4))
    x=np.arange(len(names)); width=0.35
    ax.bar(x-width/2,[results["field_level"][n]["recall"] for n in names],width,label="recall")
    ax.bar(x+width/2,[results["field_level"][n]["fpr"] for n in names],width,label="FPR")
    ax.set_xticks(x); ax.set_xticklabels([n.replace(" ","\n") for n in names],rotation=0,fontsize=8)
    ax.legend(); ax.set_title("Field-level ablation on held-out synthetic test split")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed; skipping optional chart")